# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 33.6387


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 251.75 GB
MemAvailable: 938.42 GB
Free GPU Memory (GB): 33.6387

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.


/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################



## 2. Loading Datasets

### 2.1 T-Rex

In [ ]:
import os

print("\n################################")
print("Setting up T-REX...")
print("################################\n")

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.

from datasets import load_dataset
ds = load_dataset("relbert/t_rex")
ds


## 3. FKTC Evaluation

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import os
import pandas as pd

class ResponseGenerator:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
        self.model.eval()

    def generate_response(self, query, strategy, true_answer, max_new_tokens, temperature):
        prompt = self.get_prompt(query, strategy)
        input_ids = self.tokenizer.encode(prompt, return_tensors='pt').to("cuda")
        generation_config = {
            "temperature": temperature,
            "do_sample": True,
            "top_p": 0.75,
            "top_k": 40,
            "num_beams": 5,
            "num_return_sequences": 3,
            "output_scores": True,
            "output_hidden_states": False,
            "output_attentions": False,
            "return_dict_in_generate": True
        }

        with torch.no_grad():
            outputs = self.model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens)

        output_text = self.tokenizer.decode(outputs[0][0], skip_special_tokens=True)
        output_text = self.clean_response(output_text, strategy)
        response_tokens = outputs[0].tolist()
        output_scores = outputs.scores

        is_correct, cumulative_prob, beams_with_probs, top_tokens_with_probs = self.check_answer(response_tokens[0], true_answer, output_scores, input_ids)
        return output_text, is_correct, cumulative_prob, beams_with_probs, top_tokens_with_probs

    def get_prompt(self, query, strategy):
        # Define the prompt formatting based on the selected strategy
        prompt_strategies = {
            "Fact Statement": f"{query} Fact:",
            "Completion": f"{query} The answer is:",
            "Definitive Statement": f"The answer to the question '{query}' is:",
            "True Statement": f"It is true that the answer to '{query}' is:",
            "Declarative Statement": f"{query} The fact is:",
            "Conclusive Statement": f"The final answer to '{query}' is:",
            "Resolved Statement": f"Resolved: '{query}' The answer is:",
            "Ending Completion": f"{query} The final answer is:",
            "Answer Completion": f"{query} The correct answer is:",
            "Plain Completion": f"{query} The answer:",
            "Direct Completion": f"{query} Answer:",
            "Simple Completion": f"{query} Result:",
            "Direct Answer": f"{query} Correct answer:",
            "Answer Statement": f"{query} The exact answer is:",
            "True Completion": f"{query} The true answer is:"
        }
        return prompt_strategies.get(strategy, query)

    def clean_response(self, output_text, strategy):
        # Clean the output text based on the selected strategy
        strategy_endings = {
            "Fact Statement": "Fact:",
            "Completion": "The answer is:",
            "Definitive Statement": "is:",
            "True Statement": "is:",
            "Declarative Statement": "The fact is:",
            "Conclusive Statement": "is:",
            "Resolved Statement": "is:",
            "Ending Completion": "is:",
            "Answer Completion": "The correct answer is:",
            "Plain Completion": "The answer:",
            "Direct Completion": "Answer:",
            "Simple Completion": "Result:",
            "Direct Answer": "Correct answer:",
            "Answer Statement": "The exact answer is:",
            "True Completion": "The true answer is:"
        }
        return output_text.split(strategy_endings.get(strategy, ""))[-1].strip()

    def check_answer(self, response_tokens, true_answer, output_scores, input_ids):
        generated_tokens = response_tokens[input_ids.size(1):]
        if len(generated_tokens) == 0:
            return False, None, None, None
        assert len(generated_tokens) == len(output_scores)
        
        # Generate the four variants of true_answer
        true_answer_lower = true_answer.lower()
        true_answer_title = true_answer.title()
        true_tokens_no_space_lower = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(true_answer_lower))
        true_tokens_no_space_title = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(true_answer_title))
        true_tokens_with_space_lower = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(" " + true_answer_lower))
        true_tokens_with_space_title = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(" " + true_answer_title))

        variants = [
            true_tokens_no_space_lower,
            true_tokens_no_space_title,
            true_tokens_with_space_lower,
            true_tokens_with_space_title
        ]

        def get_cumulative_probability(true_tokens, idx, output_scores):
            cumulative_prob = 1.0
            top_tokens_with_probs = []
            beams_with_probs = []

            for true_token_idx, true_token in enumerate(true_tokens):
                token_probs = []
                for beam_index, beam_scores in enumerate(output_scores[idx + true_token_idx]):
                    token_probs = torch.softmax(beam_scores, dim=-1)
                    token_prob = token_probs[true_token].item()
                    top_indices = (token_probs >= 0.1).nonzero(as_tuple=True)[0]
                    top_probs = token_probs[top_indices]
                    top_tokens = self.tokenizer.convert_ids_to_tokens(top_indices)
                    top_tokens = [token.replace("Ġ", " ") for token in top_tokens]

                    if true_token in top_indices:
                        top_tokens_with_probs.extend([(token, prob.item()) for token, prob in zip(top_tokens, top_probs)])
                        cumulative_prob *= token_prob
                        beams_with_probs.append({
                            'character_index': idx + true_token_idx,
                            'beam_index': beam_index,
                            'token': self.tokenizer.decode([true_token]),
                            'token_index': true_token,
                            'probability': token_prob
                        })
                        break

            return cumulative_prob, beams_with_probs, top_tokens_with_probs

        # Check each variant
        for variant in variants:
            for idx in range(len(generated_tokens) - len(variant) + 1):
                if generated_tokens[idx:idx + len(variant)] == variant:
                    cumulative_prob, beams_with_probs, top_tokens_with_probs = get_cumulative_probability(variant, idx, output_scores)
                    return True, cumulative_prob, beams_with_probs, top_tokens_with_probs

        return False, None, None, None

In [ ]:
strategies = [
    "Fact Statement", "Completion", "Definitive Statement", "True Statement",
    "Declarative Statement", "Conclusive Statement", "Resolved Statement",
    "Ending Completion", "Answer Completion", "Plain Completion", "Direct Completion",
    "Simple Completion", "Direct Answer", "Answer Statement", "True Completion"
]

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import os

class MonitorEvaluator:
    def __init__(self, model_name, data_dir, files, generation_config=None, max_new_tokens=15, verbose=True):
        print("Initializing MonitorEvaluator...")
        print(f"Loading model {model_name}...")
        print(f"Loading tokenizer {model_name}...")
        print(f"Loading data from {data_dir}...")
        print(f"Loading files {files}...")
        print(f"Max new tokens: {max_new_tokens}")
        self.generator = ResponseGenerator(model_name)
        self.data_dir = data_dir
        self.files = files
        self.max_new_tokens = max_new_tokens
        self.verbose = verbose
        self.generation_config = generation_config

    def load_json_data(self, filename):
        if self.verbose:
            print(f"Loading data from {filename}...")
        with open(os.path.join(self.data_dir, filename), 'r', encoding='utf8') as f:
            return [json.loads(line) for line in f.readlines()[:4]]  # Load only 4 lines per file for testing

    def evaluate(self):
        if self.verbose:
            print("Evaluating FKTC data...")
        all_results = []
        for file in self.files[:3]:  # Limit to 3 files for testing
            data = self.load_json_data(file)
            relations = data[0]['relations']
            for idx, entry in enumerate(data[1:3]):
                subject = entry['subject']
                true_object = entry['object']
                taxonomy = entry['taxonomy']
                results = []

                for relation in relations[:1]:
                    for strategy in [
                        "Fact Statement", "Completion", "Definitive Statement", "True Statement",
                        "Declarative Statement", "Conclusive Statement", "Resolved Statement",
                        "Ending Completion", "Answer Completion", "Plain Completion", "Direct Completion",
                        "Simple Completion", "Direct Answer", "Answer Statement", "True Completion"
                    ]:
                        # Evaluate original relation
                        prompt = relation.replace("[X]", subject)
                        output_text, is_correct, cumulative_prob, beams_with_probs, top_tokens_with_probs = self.generator.generate_response(prompt, strategy, true_object, self.max_new_tokens, self.generation_config.temperature)
                        results.append({
                            'file': file,
                            'entry': idx,
                            'relation': relation,
                            'prompt': prompt,
                            'subject': subject,
                            'true_object': true_object,
                            'strategy': strategy,
                            'output_text': output_text,
                            'is_correct': is_correct,
                            'cumulative_prob': cumulative_prob,
                            'beams_with_probs': beams_with_probs,
                            'top_tokens_with_probs': top_tokens_with_probs
                        })

                all_results.append(results)
        return all_results

    def save_results(self, results, output_file):
        with open(output_file, 'w', encoding='utf8') as f:
            json.dump(results, f, indent=4)

if __name__ == "__main__":
    model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    # model_name = "TinyLlama/TinyLlama_v1.1"
    # model_name = "bigscience/bloomz-560m"
    model_name = "bigscience/bloomz-1b1"
    model_name = "meta-llama/Meta-Llama-3-8B"
    # model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
    data_dir = "/nfs/students/daro/data/MONITOR/FKTC"
    files = [
        "P101-subclass.json",
        "P103-subclass.json",
        "P108-subclass.json",
        "P127-subclass.json",
        "P1376-subclass.json",
        "P1412-subclass.json",
        "P159-subclass.json",
        "P17-subclass.json",
        "P176-subclass.json",
        "P178-subclass.json",
        "P19-subclass.json",
        "P20-subclass.json",
        "P264-subclass.json",
        "P27-subclass.json",
        "P276-subclass..json",
        "P30-subclass.json",
        "P364-subclass.json",
        "P37-subclass.json",
        "P495-subclass.json",
        "P740-subclass.json"
    ]
    output_file = "evaluation_results.json"
    max_new_tokens = 15
    generation_config = {
        "temperature": 0.1,
        "top_p": 0.75,
        "top_k": 40,
        "num_beams": 5,
        "num_return_sequences": 1,
        "output_scores": True,
        "output_hidden_states": False,
        "output_attentions": False,
        "return_dict_in_generate": True
    }
    evaluator = MonitorEvaluator(
        model_name=model_name,
        data_dir=data_dir,
        files=files,
        generation_config=generation_config,
        max_new_tokens=max_new_tokens,
        verbose=False
    )
    results = evaluator.evaluate()
    for results_list in results:
        for result in results_list:
            if True:
                print(f"Prompt: {result['prompt']}")
                print(f"Answer: {result['answer']}")
                print(f"True Object: {result['true_object']}")
    print(f"Correct answers: {sum([sum([result['is_correct'] for result in results_list]) for results_list in results])}")
    print(f"Incorrect answers: {sum([sum([not result['is_correct'] for result in results_list]) for results_list in results])}")
    evaluator.save_results(results, output_file)

In [ ]:
import json
import os

class FKTCObjectChecker:
    def __init__(self, data_dir, files):
        self.data_dir = data_dir
        self.files = files

    def load_json_data(self, filename):
        with open(os.path.join(self.data_dir, filename), 'r', encoding='utf8') as f:
            return [json.loads(line) for line in f.readlines()]

    def check_objects_for_multiple_words(self):
        multi_word_objects = {}
        for file in self.files:
            data = self.load_json_data(file)
            for entry in data:
                obj = entry.get('object', "")
                if len(obj.split()) > 1:  # Check if the object contains more than one word
                    if file not in multi_word_objects:
                        multi_word_objects[file] = []
                    multi_word_objects[file].append(obj)
        
        return multi_word_objects

if __name__ == "__main__":
    data_dir = "/nfs/students/daro/data/MONITOR/FKTC"
    files = [
        "P101-subclass.json",
        "P103-subclass.json",
        "P108-subclass.json",
        "P127-subclass.json",
        "P1376-subclass.json",
        "P1412-subclass.json",
        "P159-subclass.json",
        "P17-subclass.json",
        "P176-subclass.json",
        "P178-subclass.json",
        "P19-subclass.json",
        "P20-subclass.json",
        "P264-subclass.json",
        "P27-subclass.json",
        "P276-subclass..json",
        "P30-subclass.json",
        "P364-subclass.json",
        "P37-subclass.json",
        "P495-subclass.json",
        "P740-subclass.json"
    ]

    checker = FKTCObjectChecker(data_dir, files)
    multi_word_objects = checker.check_objects_for_multiple_words()

    if multi_word_objects:
        print("Files with multi-word 'object' values:")
        for file, objects in multi_word_objects.items():
            print(f"\n{file}:")
            for obj in objects:
                print(f"  - {obj}")
    else:
        print("No multi-word 'object' values found in the files.")

In [ ]:
results = evaluator.evaluate()
for results_list in results:
    for result in results_list:
        if result['is_correct']:
            print(f"Prompt: {result['prompt']}")
            print(f"Answer: {result['answer']}")
            print(f"True Object: {result['true_object']}")

In [ ]:
print(sum([sum([result['is_correct'] for result in results_list]) for results_list in results]))
print(sum([sum([not result['is_correct'] for result in results_list]) for results_list in results]))

In [ ]:
for results_list in results:
  for result in results_list:
    if not result['is_correct']:
      print(f"Prompt: {result['prompt']}")
      print(f"Answer: {result['answer']}")
      print(f"True Object: {result['true_object']}")

In [ ]:
import transformers
import torch

model_id = "meta-llama/Meta-Llama-3-8B"
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
pipeline = transformers.pipeline(
  "text-generation", model=model_id, model_kwargs={"torch_dtype": torch.bfloat16}, device_map="cuda"
)
# output = pipeline("What is the capital city of Hungary?", max_new_tokens=15)
output = pipeline("Which city is Chandos Records's corporate headquarters located?", max_new_tokens=100)
answer = output[0]['generated_text']
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(output)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Load tokenizer and model with float16 precision
print("Loading model...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Create a pipeline for text generation
print("Creating pipeline...")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="cuda")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Perform inference with the query
print("Performing inference...")
query = "Which city is Chandos Records's corporate headquarters located?"
# query = "What is the capital city of Hungary?"
output = generator(query, max_length=200, num_return_sequences=1)
print(output)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import numpy as np

print("Loading tokenizer and model...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
# model_name = "bigscience/bloomz-560m"
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# model_name = "TinyLlama/TinyLlama_v1.1"
# model_name = "bigscience/bloomz-560m"
model_name = "bigscience/bloomz-1b1"
# model_name = "meta-llama/Meta-Llama-3-8B"
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")

# query = "Which city is Eiffel Tower located in?"
# query = "Spanish. What is the native language of Louis Jules Trochu?"
query = "Which industry does Alan Turing work in?"
# query = "What is the location of Simcoe Composite School?"
input_ids = tokenizer.encode(query, return_tensors='pt').to("cuda")

max_length = 50
generation_config = {
    "temperature": 1,
    "top_p": 0.75,
    "top_k": 40,
    "num_beams": 5,
    "num_return_sequences": 1,
    "output_scores": True,
    "output_hidden_states": False,
    "output_attentions": False,
    "return_dict_in_generate": True
}

print("Generating output...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
with torch.no_grad():
    output_ids = model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=15)

print("Decoding output...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
output_text = tokenizer.decode(output_ids[0][0], skip_special_tokens=True)
print(output_text)

In [ ]:
from transformers import AutoTokenizer
import transformers 
import torch
model = "TinyLlama/TinyLlama_v1.1"
# model = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model)
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    torch_dtype=torch.float16,
    device_map="auto",
)

sequences = pipeline(
    query,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    repetition_penalty=1.5,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=15,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")


### 3.1 All strategies

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd

class ResponseGenerator:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
        self.model.eval()

    def generate_response(self, prompt, max_new_tokens, temperature):
        input_ids = self.tokenizer.encode(prompt, return_tensors='pt').to("cuda")
        generation_config = {
            "temperature": temperature,
            "do_sample": True,
            "top_p": 0.75,
            "top_k": 40,
            "num_beams": 5,
            "num_return_sequences": 1,
            "output_scores": True,
            "output_hidden_states": False,
            "output_attentions": False,
            "return_dict_in_generate": True
        }

        with torch.no_grad():
            outputs = self.model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens)

        output_text = self.tokenizer.decode(outputs[0][0], skip_special_tokens=True)
        probabilities = self.extract_probabilities(outputs)
        return output_text, probabilities

    def extract_probabilities(self, outputs):
        probabilities = []
        for score in outputs.scores:
            probs = torch.softmax(score[0], dim=-1)
            top_prob, top_idx = torch.max(probs, dim=-1)
            probabilities.append((self.tokenizer.decode(top_idx), top_prob.item()))
        return probabilities

def apply_prompt_strategy(query, strategy):
    if strategy == "Direct Instruction":
        return f"Please answer the following question in one word.\nQuestion: {query}\nAnswer:"
    elif strategy == "Contextual Prompts":
        return f"{query} (Please answer in one word)"
    elif strategy == "Explicit Formatting":
        return f"What is the location of {query}?\nAnswer (one word):"
    elif strategy == "Question-Answer Pairs":
        return f"QSTN: What is the capital of France?\nANSR: Paris\nQSTN: What is the capital of Germany?\nANSR: Berlin\nQSTN: {query}\nANSR:"
    elif strategy == "Negative Examples":
        return f"QSTN: What is the capital of France?\nANSR: Berlin (incorrect)\nANSR: Paris (correct)\nQSTN: {query}\nANSR:"
    elif strategy == "Direct Answer Request":
        return f"Give a concise answer: {query}\nAnswer:"
    elif strategy == "Role Play":
        return f"You are a geography expert known for your concise answers.\nQuestion: {query}\nAnswer:"
    elif strategy == "Simplified Question":
        return f"Where is {query} located?\nAnswer:"
    elif strategy == "List Format":
        return f"List of schools and their locations:\n1. Harvard University - USA\n2. University of Cambridge - UK\n3. {query} -"
    elif strategy == "Multiple Choice":
        return f"Select the correct location of {query}:\nA) USA\nB) UK\nC) Canada\nAnswer:"
    elif strategy == "Fill-in-the-Blank":
        return f"{query} is located in _____.\nAnswer:"
    elif strategy == "Structured Answer Prompt":
        return f"Question: {query}\nAnswer (one word):"
    else:
        return query

# Example usage:
model_names = [
    "bigscience/bloomz-560m",
    "bigscience/bloomz-1b1",
    "openai-community/gpt2-large",
    "EleutherAI/gpt-neo-1.3B",
    "TinyLlama/TinyLlama_v1.1",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    #"meta-llama/Meta-Llama-3-8B",
    #"meta-llama/Meta-Llama-3-8B-Instruct",
]
queries = [
    "What is the location of Simcoe Composite School?",
    "What is Alan Turing's area of expertise?",
    "What is the native language of Louis Jules Trochu?",
    "What is the capital city of Italy?",
    "What is the main ingredient in sushi?"
]
true_answers = [
    "Canada",
    "logic",
    "French",
    "Rome",
    "rice"
]
max_new_tokens_list = [5, 15, 25]
temperature_list = [0.1, 1]
strategies = [
    # "Direct Instruction",
    # "Contextual Prompts",
    # "Explicit Formatting",
    "Question-Answer Pairs",
    "Negative Examples",
    # "Direct Answer Request",
    # "Role Play",
    # "Simplified Question",
    # "List Format",
    # "Multiple Choice",
    # "Fill-in-the-Blank",
    # "Structured Answer Prompt"
]

# Initialize a list to store the results
results = []

for model_name in model_names:
    generator = ResponseGenerator(model_name)
    for query, true_answer in zip(queries, true_answers):
        for max_new_tokens in max_new_tokens_list:
            for temperature in temperature_list:
                for strategy in strategies:
                    print(f"Model: {model_name}, Query: {query}, Strategy: {strategy}, Max New Tokens: {max_new_tokens}, Temperature: {temperature}")
                    prompt = apply_prompt_strategy(query, strategy)
                    output_text, probabilities = generator.generate_response(prompt, max_new_tokens, temperature)
                    if output_text.startswith(prompt):
                        output_text = output_text[len(prompt):].strip()
                    results.append({
                        "Model": model_name,
                        "Query": query,
                        "Strategy": strategy,
                        "Max New Tokens": max_new_tokens,
                        "Temperature": temperature,
                        "True Answer": true_answer,
                        "Output": output_text,
                        "Probabilities": probabilities
                    })

# Convert the results list to a pandas DataFrame
df = pd.DataFrame(results)

# Print the DataFrame
print(df)

# Export the DataFrame to an Excel file
df.to_excel("results_07_25_qa_negative.xlsx", index=False)